# Veículos Eletrificados vs Combustíveis

Este notebook reúne dados públicos sobre **vendas de veículos eletrificados** (híbridos e elétricos) e **consumo de combustíveis** (gasolina C e etanol hidratado) no Brasil entre 2020 e 2024. O objetivo é simples: **entender se o crescimento dos carros eletrificados está reduzindo o consumo de combustíveis**.

### Racional da análise

1. Como evoluíram as vendas de veículos eletrificados
2. Como se comportou o consumo de gasolina e etanol no mesmo período
3. Se existe alguma relação entre essas duas tendências
4. Uma projeção simples para os próximos anos

### Fontes

- **Veículos eletrificados:** Associação Brasileira do Veículo Elétrico (ABVE)
- **Consumo de combustíveis:** Agência Nacional do Petróleo (ANP)

> ℹ️ São apenas 5 anos de dados anuais (2020 a 2024), então as conclusões servem como **ponto de partida para discussão**, não como prova definitiva de causa e efeito.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score,    
)

#Configuração visual
pd.options.display.float_format = '{:.2f}'.format

In [0]:

df_abve = pd.read_csv('../../data/abve_eletrificados_2020_2024.csv',
                      sep=';')
df_gasolina = pd.read_csv('../../data/anp_gasolina_c_2020_2024.csv', 
                          sep=';')
df_etanol = pd.read_csv('../../data/anp_etanol_hidratado_2020_2024.csv', 
                         sep=';')

#Consolidação anual

df_abve_anual = (df_abve.groupby('ano', as_index=False)
                 ['total_eletrificados'].sum())
df_gasolina_anual = (df_gasolina.groupby('ANO', as_index=False)
                 ['VENDAS'].sum())
df_etanol_anual = (df_etanol.groupby('ANO', as_index=False)
                 ['VENDAS'].sum())

In [0]:
df_gasolina_anual = df_gasolina_anual.rename(columns={'ANO': 'ano', 'VENDAS':'gasolina_litros'})

df_etanol_anual = df_etanol_anual.rename(columns={'ANO': 'ano', 'VENDAS':'etanol_litros'})
                                         
print(df_gasolina_anual.columns.tolist())
print(df_etanol_anual.columns.tolist())

In [0]:
df_gold = (df_abve_anual.merge(df_gasolina_anual, on='ano', how='inner').merge(df_etanol_anual, on='ano', how='inner'))

df_gold['gasolina_litros'] = df_gold['gasolina_litros'] / 1_000_000_000
df_gold['etanol_litros'] = df_gold['etanol_litros'] / 1_00000_0000


In [0]:
#Visibilidade por segmento
df_gold['eletrificados_crescimento'] = (df_gold['total_eletrificados'].pct_change() * 100)

df_gold['gasolina_crescimento'] = (df_gold['gasolina_litros'].pct_change() * 100)

df_gold['etanol_crescimento'] = (df_gold['etanol_litros'].pct_change() * 100)


In [0]:
df_gold_exibicao = df_gold.copy()

df_gold_exibicao['total_eletrificados'] = df_gold_exibicao['total_eletrificados'].apply(lambda x: f'{x:,.0f}')

df_gold_exibicao.round(2)


## Evolução de Veículos Eletrificados no Brasil (2020-2024)

Em poucas palavras: **os veículos eletrificados cresceram muito e rápido**.

- Em 2020, foram vendidas cerca de **19,7 mil** unidades
- Em 2024, esse número saltou para **177,4 mil** — quase **9 vezes mais**
- O crescimento ficou ainda mais forte em **2023 e 2024**, quando as vendas praticamente dobraram de um ano para o outro

### Por que isso aconteceu?

Esse salto reflete uma combinação de fatores: mais modelos disponíveis no mercado, políticas de incentivo, queda nos preços das baterias e maior oferta de infraestrutura de recarga. O ano de 2023 marcou um **divisor de águas** — as vendas passaram de 49 mil para quase 94 mil unidades em apenas um ano.

In [0]:
#Evolução dos veículos eletrificados Brasil (2020-2024)
plt.figure(figsize=(10, 5))
plt.title('Evolução de Veículos Eletrificados no Brasil (2020-2024)')

plt.plot(df_gold['ano'], df_gold['total_eletrificados'],
         marker='o',
         linewidth=2,
         color='#0045B5'
)

plt.xlabel('Ano')
plt.ylabel('Quantidade de Veículos')
plt.xticks(df_gold['ano'])
plt.grid(alpha=0.3)
plt.show()

## Evolução do Consumo de Combustíveis no Brasil (2020-2024)

Em resumo: **o consumo de combustíveis se manteve estável**, sem queda significativa — mesmo com os eletrificados crescendo tanto.

### Gasolina C

- O consumo **subiu até 2023**, atingindo o pico de **46 bilhões de litros**
- Em **2024 houve uma leve queda** (44,4 bilhões), mas nada dramático
- Pode ser o primeiro sinal de impacto dos eletrificados, mas ainda é cedo para afirmar

### Etanol Hidratado

- Teve **idos e vindas** ao longo do período
- Caiu bastante entre 2020 e 2022 (de 19,3 para 15,5 bilhões de litros)
- **Recuperou-se em 2024**, chegando a 21,7 bilhões de litros

### Por que as vendas de combustíveis se mantiveram estáveis?

Simples: os veículos eletrificados ainda são uma **fatia muito pequena** da frota total de carros no Brasil. Além disso, a economia se recuperou após a pandemia e o mercado automotivo continuou crescendo, o que mantém a demanda por combustíveis.

In [0]:
#Evolução do consumo de combustíveis no Brasil (2020-2024)
plt.figure(figsize=(10, 5))
plt.title('Evolução do Consumo de Combustíveis no Brasil (2020-2024)')

plt.plot(df_gold['ano'], df_gold['gasolina_litros'],
         marker='o',
         linewidth=2,
         color='#0045B5',
         label='Gasolina C'
)

plt.plot(df_gold['ano'], df_gold['etanol_litros'],
         marker='o',
         linewidth=2,
         color='#FF9900',
         label='Etanol Hidratado'
)

plt.xlabel('Ano')
plt.ylabel('Bi Litros')
plt.xticks(df_gold['ano'])
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Correlação entre Veículos Eletrificados e Combustíveis

O mapa de calor mostra uma **correlação positiva** — ou seja, quando as vendas de eletrificados subiram, o consumo de combustíveis também subiu:

- Gasolina C: **0,71** (correlação forte)
- Etanol Hidratado: **0,60** (correlação moderada)

### Mas atenção: correlação não é causalidade!

Isso **não significa** que os eletrificados estão fazendo o consumo de combustíveis aumentar. O que aconteceu é que **ambos cresceram ao mesmo tempo**, mas por motivos diferentes:

- Os eletrificados cresceram por causa de incentivos e oferta de novos modelos
- Os combustíveis se mantiveram altos por causa da recuperação da economia e do aumento da frota total


In [0]:
# Correlação entre veículos eletrificados e combustíveis
cor = df_gold[['total_eletrificados', 'gasolina_litros', 'etanol_litros']].corr()

sns.heatmap(cor, annot=True,
             cmap='Blues',
            fmt='.2f')

plt.title("Correlação entre Veículos Eletrificados e Combustíveis")
plt.show()

## Projeções Futuras (2026-2027)

Com base nos dados de 2020 a 2024, usamos uma regressão linear simples para projetar as tendências de vendas de eletrificados e consumo de combustíveis para 2026 e 2027. É uma projeção simples, que assume que a tendência observada se mantém.

In [0]:
#Projeção de indicadores para 2026-2027
#Estou usando dados até 2024, pois os dados públicos disponíveis, não contemplaam 2025 e 2026. Nesse caso optei projetar períodos a frente, considerando o ano ainda corrente, 2026 e a ano futuro, 2027.

def projetar_tendencia_linear(df, coluna, ano_futuros = [2026, 2027]):
    X = df[['ano']]
    y = df[coluna]

    modelo = LinearRegression()
    modelo.fit(X, y)

    X_futuro = pd.DataFrame({
        'ano': ano_futuros
    })
    previsao_futura = modelo.predict(X_futuro)
    previsao_historica = modelo.predict(X)

    metricas = {'variavel': coluna,
                'MAE': mean_absolute_error(y, previsao_historica),
                'RMSE': np.sqrt(mean_squared_error(y, previsao_historica)),
                'R2': r2_score(y, previsao_historica)}
    
    resultado = X_futuro.copy()
    resultado['coluna'] = previsao_futura

    return modelo, resultado, metricas


In [0]:
#Execução de projeções

modelo_eletrificado, prev_eletrificados, metricas_eletrificadas = (
    projetar_tendencia_linear(df_gold, 'total_eletrificados')
)
modelo_gasolina, prev_gasolina, metricas_gasolina = (
    projetar_tendencia_linear(df_gold, 'gasolina_litros')
)
modelo_etanol, prev_etanol, metricas_etanol = (
    projetar_tendencia_linear(df_gold, 'etanol_litros')
    )


In [0]:
#Consolidação das projeções

df_previsao.columns = [
    'ano',
    'eletrificados_previstos',
    'gasolina_prevista_litros',
    'etanol_previsto__litros'
]

In [0]:
plt.figure(figsize=(10, 5))

plt.plot(
    df_previsao['ano'],
    df_previsao['eletrificados_previstos'],
    marker='o',
    linewidth=2,
    label='Veículos Eletrificados',
)
plt.title('Projeção de Veículos Eletrificados (2026-2027)')
plt.xlabel('Ano')
plt.ylabel('Quantidade de Veículos')
plt.grid(True)
plt.legend()

plt.show()

## Evolução e Projeção de Veículos Eletrificados

O gráfico abaixo junta o histórico de vendas (2020-2024) com a projeção (2026-2027), permitindo visualizar a tendência de crescimento contínua dos eletrificados.

In [0]:
#Histórico + Projeção de Veículos Eletrificados

plt.figure(figsize=(10, 5))

#Histórico
plt.plot(
    df_gold['ano'],
    df_gold['total_eletrificados'],
    marker='o',
    linewidth=2,
    label='Histórico (2020-2024)',
)

#Projeção
plt.plot(
    df_previsao['ano'],
    df_previsao['eletrificados_previstos'],
    marker='o',
    linewidth=2,
    label='Projeção (2026-2027)',
)

plt.title('Evolução e Projeção de Veículos Eletrificados')
plt.xlabel('Ano')
plt.ylabel('Quantidade de Veículos')
plt.grid(True)
plt.legend()

plt.show()

## Avaliação Final

### Pontos de atenção

- São apenas **5 anos de dados** — pouco para tirar conclusões definitivas
- O período inclui a pandemia, que trouxe efeitos atípicos
- Os eletrificados representam **menos de 0,5%** da frota circulante
- Um carro elétrico comprado hoje **demora anos** para impactar o consumo total

### Conclusão

Os resultados **não mostram uma redução consistente** do consumo de combustíveis associada ao crescimento dos eletrificados. São **indícios exploratórios**, não relações de causa e efeito.

### Para uma análise mais completa no futuro

- Mais anos de dados (série histórica mais longa)
- Dados mensais ou trimestrais (não apenas anuais)
- Considerar fatores como preço dos combustíveis e renda da população
- Analisar a **frota circulante** total, não apenas as vendas de cada ano